In [ ]:
import torch
from dinosaw.helpers import ModelTypes, model_names, get_models,get_features
from dinosaw.utils import do_2D_pca

import numpy as np
from PIL import Image

import matplotlib.pyplot as plt

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = 'cuda:0'
half = False

flash attention installed


In [ ]:
enabled_models: tuple[ModelTypes, ...] = ('dv2', 'dvt', 'alibi_dv2', 'alibi_dv2_cb', 'nope')
models = get_models(enabled_models, "../../trained_models", DEVICE, half)

In [3]:
SF = 1
# img_fname = "micro/NMC_2D_crop.png"\
# img_fname = "tests/black_square_518.png"
# img_fname = "tests/diff_shapes_518.png"
img_fname = "tests/default_image.jpg"
# img_fname = "cat.jpg"
# img_fname = "tests/z_c_w.png"
# img_fname = "micro/NMC_2D_less_wide_crop.png"
_img = Image.open(f"../../images/{img_fname}").convert("RGB")
_img = _img.resize((int(SF * _img.width), int(SF * _img.height)), Image.LANCZOS)

In [4]:
features = {}
features_reduced = {}

In [ ]:
for model_key in enabled_models:
    model = models[model_key]
    feats = get_features(model, _img, False, False, device=DEVICE)
    features[model_key] = feats

In [6]:
for model_key in enabled_models:
    feats = features[model_key]
    features_reduced[model_key] = do_2D_pca(feats, 3, post_norm='minmax')

In [9]:
%%capture
fig, axs = plt.subplots(nrows=1 + len(enabled_models), ncols=1, figsize=(25, 25))

axs[0].imshow(_img)
axs[0].set_axis_off()
for i, model_key in enumerate(enabled_models):
    feats_red = features_reduced[model_key]
    ax = axs[i + 1]
    ax.set_title(model_names[model_key], fontsize=24)
    ax.imshow(feats_red)
    ax.set_axis_off()
plt.tight_layout()